# NetCDF DEM Extraction within a Buffer

This notebook, generated by Claude AI, demonstrates how to extract DEM data within a specified buffer around coordinates from NetCDF files.

## Use Case
- You have a NetCDF file with latitude/longitude coordinates
- You want to extract all DEM data within 1 degree (or any buffer) of a specific point
- Handle coordinate system differences between NetCDF and GeoTIFF automatically

## Prerequisites
```bash
pip install xarray rasterio shapely geopandas pyproj matplotlib numpy
```

In [ ]:
# Import required libraries
import xarray as xr
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box
import geopandas as gpd
from pyproj import CRS, Transformer
import os

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")

## Step 1: Set File Paths

Update these paths to point to your NetCDF and DEM files:

In [ ]:
# UPDATE THESE PATHS TO YOUR FILES
netcdf_path = "path/to/your/data.nc"
dem_path = "path/to/your/dem.tif"

# Check if files exist
print("File Status:")
print(f"NetCDF: {'✓ Found' if os.path.exists(netcdf_path) else '✗ Not found'} - {netcdf_path}")
print(f"DEM: {'✓ Found' if os.path.exists(dem_path) else '✗ Not found'} - {dem_path}")

if not os.path.exists(netcdf_path) or not os.path.exists(dem_path):
    print("\n⚠️  Please update the file paths above before continuing")

## Step 2: Explore NetCDF Structure

Let's examine the NetCDF file to understand its structure and find the coordinate variables:

In [ ]:
# Open and explore NetCDF structure
try:
    ds = xr.open_dataset(netcdf_path)
    
    print("=== NETCDF OVERVIEW ===")
    print(f"Dimensions: {dict(ds.dims)}")
    print(f"\nCoordinates: {list(ds.coords.keys())}")
    print(f"\nData variables: {list(ds.data_vars.keys())}")
    
    # Show basic info about the dataset
    print("\n=== DATASET INFO ===")
    print(ds.info())
    
    # Look for latitude/longitude variables
    print("\n=== COORDINATE VARIABLES ===")
    lat_vars = [v for v in ds.variables if 'lat' in v.lower()]
    lon_vars = [v for v in ds.variables if 'lon' in v.lower()]
    
    print(f"Potential latitude variables: {lat_vars}")
    print(f"Potential longitude variables: {lon_vars}")
    
    # Show coordinate info if found
    if lat_vars:
        lat_var = lat_vars[0]
        print(f"\nLatitude variable '{lat_var}':")
        print(f"  Shape: {ds[lat_var].shape}")
        print(f"  Range: {float(ds[lat_var].min()):.4f} to {float(ds[lat_var].max()):.4f}")
        
    if lon_vars:
        lon_var = lon_vars[0]
        print(f"\nLongitude variable '{lon_var}':")
        print(f"  Shape: {ds[lon_var].shape}")
        print(f"  Range: {float(ds[lon_var].min()):.4f} to {float(ds[lon_var].max()):.4f}")
        
except Exception as e:
    print(f"Error opening NetCDF: {e}")
    ds = None

## Step 3: Extract Coordinates from NetCDF

Now let's create a function to extract coordinates and try different methods:

In [ ]:
def get_netcdf_coordinates(ds, method='center', **kwargs):
    """
    Extract coordinates from NetCDF dataset using different methods
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Opened NetCDF dataset
    method : str
        'center', 'index', or 'criteria'
    **kwargs : dict
        Additional parameters for specific methods
    """
    # Find latitude/longitude variables
    lat_var = None
    lon_var = None
    
    # Common naming patterns for lat/lon
    lat_names = ['latitude', 'lat', 'Latitude', 'LAT', 'LATITUDE']
    lon_names = ['longitude', 'lon', 'Longitude', 'LON', 'LONGITUDE']
    
    for name in lat_names:
        if name in ds.variables:
            lat_var = name
            break
    
    for name in lon_names:
        if name in ds.variables:
            lon_var = name
            break
    
    if not lat_var or not lon_var:
        raise ValueError(f"Could not find lat/lon variables. Available: {list(ds.variables.keys())}")
    
    print(f"Using latitude variable: '{lat_var}'")
    print(f"Using longitude variable: '{lon_var}'")
    
    lat_array = ds[lat_var].values
    lon_array = ds[lon_var].values
    
    if method == 'center':
        # Extract center coordinate
        if lat_array.ndim == 1:
            center_lat_idx = len(lat_array) // 2
            center_lon_idx = len(lon_array) // 2
            lat = lat_array[center_lat_idx]
            lon = lon_array[center_lon_idx]
        else:
            center_indices = tuple(s // 2 for s in lat_array.shape)
            lat = lat_array[center_indices]
            lon = lon_array[center_indices]
        
        print(f"Selected center coordinate: ({float(lat):.4f}, {float(lon):.4f})")
        
    elif method == 'index':
        # Extract by specific index
        target_index = kwargs.get('index', (0, 0))
        
        if lat_array.ndim == 1:
            lat = lat_array[target_index[0]]
            lon = lon_array[target_index[1]]
        else:
            lat = lat_array[target_index]
            lon = lon_array[target_index]
        
        print(f"Selected coordinate by index {target_index}: ({float(lat):.4f}, {float(lon):.4f})")
        
    elif method == 'criteria':
        # Extract by data criteria
        var_name = kwargs.get('variable')
        min_val = kwargs.get('min_value')
        max_val = kwargs.get('max_value')
        
        if not all([var_name, min_val is not None, max_val is not None]):
            raise ValueError("For criteria method, need 'variable', 'min_value', and 'max_value'")
        
        data_var = ds[var_name]
        mask_condition = (data_var >= min_val) & (data_var <= max_val)
        
        # Get coordinates where condition is met
        lat_subset = lat_array[mask_condition]
        lon_subset = lon_array[mask_condition]
        
        if len(lat_subset) > 0:
            lat = float(lat_subset.flat[0])
            lon = float(lon_subset.flat[0])
            print(f"Selected coordinate by criteria {var_name}=[{min_val}, {max_val}]: ({lat:.4f}, {lon:.4f})")
        else:
            raise ValueError("No coordinates found matching criteria")
    
    return float(lat), float(lon)

# Test coordinate extraction if NetCDF is available
if ds is not None:
    try:
        # Method 1: Center point
        print("\n=== METHOD 1: CENTER COORDINATE ===")
        lat_center, lon_center = get_netcdf_coordinates(ds, method='center')
        
        # Method 2: Specific index (adjust as needed)
        print("\n=== METHOD 2: BY INDEX ===")
        lat_index, lon_index = get_netcdf_coordinates(ds, method='index', index=(10, 20))
        
        # Method 3: By criteria (uncomment if you have a data variable to filter by)
        # print("\n=== METHOD 3: BY CRITERIA ===")
        # lat_criteria, lon_criteria = get_netcdf_coordinates(
        #     ds, method='criteria', 
        #     variable='temperature',  # Replace with your variable name
        #     min_value=15, 
        #     max_value=25
        # )
        
        # Use center coordinate for the rest of the workflow
        selected_lat, selected_lon = lat_center, lon_center
        
    except Exception as e:
        print(f"Error extracting coordinates: {e}")
        selected_lat, selected_lon = None, None
else:
    selected_lat, selected_lon = None, None

## Step 4: Create Bounding Box

Create a bounding box around the selected coordinate with a specified buffer:

In [ ]:
def create_bounding_box(lat, lon, buffer_degrees=1.0):
    """
    Create a bounding box around a point with specified buffer in degrees
    """
    minx = lon - buffer_degrees
    maxx = lon + buffer_degrees
    miny = lat - buffer_degrees
    maxy = lat + buffer_degrees
    
    bounds = (minx, miny, maxx, maxy)
    bbox = box(minx, miny, maxx, maxy)
    
    return bbox, bounds

# Create bounding box if we have coordinates
if selected_lat is not None and selected_lon is not None:
    buffer_size = 1.0  # degrees - adjust as needed
    bbox, bounds = create_bounding_box(selected_lat, selected_lon, buffer_size)
    
    print(f"=== BOUNDING BOX CREATION ===")
    print(f"Center point: ({selected_lat:.4f}, {selected_lon:.4f})")
    print(f"Buffer size: {buffer_size}°")
    print(f"Bounding box: {bounds}")
    print(f"Box size: {2*buffer_size:.1f}° x {2*buffer_size:.1f}°")
    
    # Calculate approximate area (rough estimate)
    lat_km_per_degree = 111.0  # km per degree latitude
    lon_km_per_degree = 111.0 * np.cos(np.radians(selected_lat))  # km per degree longitude
    
    area_km2 = (2 * buffer_size * lat_km_per_degree) * (2 * buffer_size * lon_km_per_degree)
    print(f"Approximate area: {area_km2:,.0f} km²")
    
else:
    print("No coordinates available to create bounding box")
    bbox, bounds = None, None

## Step 5: Explore DEM Properties

Before extracting, let's examine the DEM file:

In [ ]:
# Examine DEM properties
try:
    with rasterio.open(dem_path) as src:
        print("=== DEM PROPERTIES ===")
        print(f"Dimensions: {src.width} x {src.height}")
        print(f"Number of bands: {src.count}")
        print(f"Data type: {src.dtypes[0]}")
        print(f"CRS: {src.crs}")
        print(f"Bounds: {src.bounds}")
        print(f"Resolution: {src.res}")
        print(f"NoData value: {src.nodata}")
        
        # Store properties for later use
        dem_crs = src.crs
        dem_bounds = src.bounds
        dem_resolution = src.res
        
        # Check if bounding box overlaps with DEM
        if bounds is not None:
            bbox_minx, bbox_miny, bbox_maxx, bbox_maxy = bounds
            dem_minx, dem_miny, dem_maxx, dem_maxy = dem_bounds
            
            # Check overlap (assuming both in geographic coordinates)
            overlap_x = max(0, min(bbox_maxx, dem_maxx) - max(bbox_minx, dem_minx))
            overlap_y = max(0, min(bbox_maxy, dem_maxy) - max(bbox_miny, dem_miny))
            
            print(f"\n=== SPATIAL OVERLAP CHECK ===")
            print(f"Bounding box bounds: {bounds}")
            print(f"DEM bounds: {dem_bounds}")
            
            if overlap_x > 0 and overlap_y > 0:
                print(f"✓ Overlap detected: {overlap_x:.4f}° x {overlap_y:.4f}°")
                has_overlap = True
            else:
                print(f"✗ No overlap detected")
                has_overlap = False
                
except Exception as e:
    print(f"Error reading DEM: {e}")
    dem_crs = None
    has_overlap = False

## Step 6: Extract DEM Subset

Now let's extract the DEM data within our bounding box:

In [ ]:
def extract_dem_with_bbox(dem_path, bbox, handle_crs_mismatch=True):
    """
    Extract DEM data within bounding box, handling coordinate system differences
    """
    with rasterio.open(dem_path) as src:
        src_crs = src.crs
        bbox_crs = CRS.from_epsg(4326)  # Assume bbox is in WGS84
        
        print(f"DEM CRS: {src_crs}")
        print(f"Bbox CRS: {bbox_crs}")
        
        # Handle coordinate system mismatch
        if handle_crs_mismatch and src_crs != bbox_crs:
            print("Transforming bounding box to match DEM coordinate system...")
            
            transformer = Transformer.from_crs(bbox_crs, src_crs, always_xy=True)
            
            # Transform bbox coordinates
            minx, miny, maxx, maxy = bbox.bounds
            transformed_coords = []
            
            # Transform all four corners
            corners = [(minx, miny), (maxx, miny), (maxx, maxy), (minx, maxy)]
            for x, y in corners:
                tx, ty = transformer.transform(x, y)
                transformed_coords.extend([tx, ty])
            
            # Create new bbox in DEM's coordinate system
            tx_minx = min(transformed_coords[0::2])
            tx_maxx = max(transformed_coords[0::2])
            tx_miny = min(transformed_coords[1::2])
            tx_maxy = max(transformed_coords[1::2])
            
            transformed_bbox = box(tx_minx, tx_miny, tx_maxx, tx_maxy)
            print(f"Transformed bbox: ({tx_minx:.2f}, {tx_miny:.2f}, {tx_maxx:.2f}, {tx_maxy:.2f})")
            
            mask_geometry = transformed_bbox
        else:
            mask_geometry = bbox
        
        # Extract subset using mask
        try:
            extracted_data, extracted_transform = mask(
                src, [mask_geometry], crop=True, filled=False, nodata=src.nodata
            )
            
            extracted_dem = extracted_data[0]  # Get first band
            
            print(f"\n=== EXTRACTION RESULTS ===")
            print(f"Original DEM shape: {src.height} x {src.width}")
            print(f"Extracted shape: {extracted_dem.shape}")
            
            # Calculate extracted bounds
            extracted_bounds = rasterio.transform.array_bounds(
                extracted_dem.shape[0], extracted_dem.shape[1], extracted_transform
            )
            print(f"Extracted bounds: {extracted_bounds}")
            
            return extracted_dem, extracted_transform, src_crs, extracted_bounds
            
        except Exception as e:
            print(f"Error during extraction: {e}")
            return None, None, None, None

# Perform extraction if we have all necessary components
if bbox is not None and has_overlap:
    try:
        extracted_dem, extracted_transform, extracted_crs, extracted_bounds = extract_dem_with_bbox(
            dem_path, bbox, handle_crs_mismatch=True
        )
        
        if extracted_dem is not None:
            print(f"\n✓ Extraction successful!")
            extraction_success = True
        else:
            print(f"\n✗ Extraction failed")
            extraction_success = False
            
    except Exception as e:
        print(f"Error during extraction: {e}")
        extraction_success = False
        extracted_dem = None
else:
    print("Cannot perform extraction - missing bbox or no spatial overlap")
    extraction_success = False
    extracted_dem = None

## Step 7: Analyze Extracted DEM

Let's examine the statistics of our extracted DEM data:

In [ ]:
if extraction_success and extracted_dem is not None:
    print("=== EXTRACTED DEM ANALYSIS ===")
    
    # Handle different types of nodata values
    if np.issubdtype(extracted_dem.dtype, np.integer):
        # Integer DEM - might have specific nodata value
        with rasterio.open(dem_path) as src:
            nodata_val = src.nodata
        if nodata_val is not None:
            valid_mask = extracted_dem != nodata_val
        else:
            valid_mask = ~np.isnan(extracted_dem.astype(float))
    else:
        # Float DEM - check for NaN
        valid_mask = ~np.isnan(extracted_dem)
    
    valid_data = extracted_dem[valid_mask]
    
    print(f"Extracted array shape: {extracted_dem.shape}")
    print(f"Total pixels: {extracted_dem.size:,}")
    print(f"Valid pixels: {len(valid_data):,}")
    print(f"NoData pixels: {extracted_dem.size - len(valid_data):,}")
    print(f"Data coverage: {len(valid_data)/extracted_dem.size*100:.1f}%")
    
    if len(valid_data) > 0:
        print(f"\n=== ELEVATION STATISTICS ===")
        print(f"Min elevation: {np.min(valid_data):.1f} m")
        print(f"Max elevation: {np.max(valid_data):.1f} m")
        print(f"Mean elevation: {np.mean(valid_data):.1f} m")
        print(f"Median elevation: {np.median(valid_data):.1f} m")
        print(f"Std deviation: {np.std(valid_data):.1f} m")
        print(f"Elevation range: {np.ptp(valid_data):.1f} m")
        
        # Calculate some percentiles
        percentiles = [10, 25, 75, 90]
        percs = np.percentile(valid_data, percentiles)
        print(f"\n=== ELEVATION PERCENTILES ===")
        for p, val in zip(percentiles, percs):
            print(f"{p}th percentile: {val:.1f} m")
            
        # Store stats for visualization
        dem_stats = {
            'min': np.min(valid_data),
            'max': np.max(valid_data),
            'mean': np.mean(valid_data),
            'std': np.std(valid_data),
            'valid_pixels': len(valid_data),
            'total_pixels': extracted_dem.size
        }
    else:
        print("\n⚠️  No valid elevation data found in extracted area")
        dem_stats = None
else:
    print("No extracted DEM data to analyze")
    dem_stats = None

## Step 8: Visualize Extracted DEM

Finally, let's visualize the extracted DEM data:

In [ ]:
if extraction_success and extracted_dem is not None:
    plt.figure(figsize=(12, 8))
    plt.imshow(extracted_dem, cmap='terrain', vmin=np.nanmin(extracted_dem), vmax=np.nanmax(extracted_dem))
    plt.colorbar(label='Elevation (m)')
    plt.title('Extracted DEM')
    plt.xlabel('Pixel X')
    plt.ylabel('Pixel Y')
    plt.grid(False)
    plt.show()

else:
    print("No DEM data to visualize")